# Project 1 — Paired Transcriptomic Analysis of T1 vs T3 Samples

## Research question

How does the transcriptomic program differ between matched T1 and T3 samples from the same patients?

The analysis focuses on:

1. sample structure and quality,
2. paired differential expression,
3. pathway-level changes,
4. convergence of GO and KEGG evidence,
5. representative candidate genes,
6. patient-level paired validation.

The analysis uses 24 matched patients, giving 48 samples in total:
24 T1 samples and 24 T3 samples.

## Why a paired design?

Each patient contributes both a T1 and a T3 sample.

Therefore, samples from the same patient are not independent. A simple
T3-vs-T1 comparison would ignore patient-specific baseline differences.

The statistical model therefore includes:

    ~ patient_id + timepoint

with T1 as the reference level.

The coefficient of interest is:

    timepoint_T3_vs_T1

This asks:

> After accounting for patient-to-patient differences, which transcriptomic
> features are associated with T3 relative to T1?

## Reproducible analysis structure

The project separates:

- `scripts/` — executable analysis pipeline
- `notebooks/` — reasoning, visualization and interpretation
- `results/` — generated outputs
- `docs/` — final scientific record

The main analysis scripts are:

01_extract_T1_T3.sh
02_qc_filter_normalize.R
03_deseq2.R
04_diagnose_DE.R
05_annotation_and_ranking.R
06_gsea_and_pathways.R
07_final_candidate_validation_and_figures.R
08_final_figures.R

In [ ]:
library(DESeq2)
library(ggplot2)
library(pheatmap)
library(clusterProfiler)
library(org.Hs.eg.db)
library(AnnotationDbi)

## Dataset preparation

The processed GEO expression matrix contained approximately 19,503 genes.

After removal of one invalid/duplicate row, 19,502 genes remained.

Low-expression filtering removed 5,599 genes, leaving:

    13,903 genes

for the paired DESeq2 analysis.

This filtering reduces the multiple-testing burden and removes genes with
insufficient expression information.

## Normalization

DESeq2 estimates sample-specific size factors so that differences in sequencing
depth and library composition do not become confused with biological changes.

The estimated normalization factors ranged approximately from:

    0.555 to 2.075

with a median close to 1.

This indicates that the samples required meaningful but not extreme scaling
adjustments.

## Exploratory sample structure

Variance-stabilizing transformation (VST) was used before visualization.

PCA summarizes the largest axes of variation across samples.

In this dataset:

- PC1 explains approximately 25% of the variance.
- PC2 explains approximately 12%.

T1 and T3 samples show substantial overlap, while patient-to-patient
heterogeneity remains visible.

This is important because it provides a visual reason for using the paired
design: biological variation between patients is substantial.

In [ ]:
vsd <- vst(dds, blind = FALSE)

pca_df <- plotPCA(
  vsd,
  intgroup = c("timepoint", "patient_id"),
  returnData = TRUE
)

ggplot(pca_df, aes(PC1, PC2, label = patient_id, shape = timepoint)) +
  geom_point(size = 3) +
  labs(
    title = "PCA of paired T1/T3 samples",
    x = paste0("PC1 (", round(attr(pca_df, "percentVar")[1] * 100), "%)"),
    y = paste0("PC2 (", round(attr(pca_df, "percentVar")[2] * 100), "%)")
  ) +
  theme_classic()

## Differential expression

DESeq2 models count variability using a negative binomial framework.

The paired model is:

    ~ patient_id + timepoint

The T3-vs-T1 coefficient is interpreted as the change associated with T3
after accounting for each patient's baseline.

The analysis included 13,903 genes.

The result was striking:

    0 genes with padj < 0.10
    0 genes with padj < 0.05

Therefore, the dataset does not support a conclusion that individual genes
show genome-wide statistically significant differential expression at the
chosen FDR thresholds.

## Why continue when there are no significant individual genes?

A biological transition can involve coordinated but individually modest changes
across many genes.

Testing genes one-by-one asks:

> Is this individual gene strong enough to survive multiple testing?

Pathway analysis asks a different question:

> Are genes belonging to a biologically related process systematically shifted
> toward one end of the ranked transcriptome?

Therefore, the absence of genome-wide significant individual genes does not
automatically imply the absence of biological structure.

The next analysis therefore uses the complete ranked gene list rather than
arbitrarily selecting only genes with p < 0.05.

## Ranked gene-set analysis

Genes were ranked using the signed DESeq2 Wald statistic.

Interpretation:

- positive statistic → T3-associated direction
- negative statistic → T1-associated direction

This preserves information across the entire dataset.

GO biological-process GSEA identified extensive pathway-level enrichment after
multiple-testing correction.

However, the GO ontology contains many overlapping and hierarchical terms.
Therefore, 863 significant GO terms should not be interpreted as 863
independent biological discoveries.

Redundancy reduction was performed using `simplify()`.

## GO biological-process result

The dominant T3-associated signal was related to:

- chromosome segregation,
- mitotic processes,
- kinetochore attachment,
- spindle checkpoint signaling,
- DNA replication,
- DNA repair.

Representative high-NES processes included mitotic sister chromatid segregation,
nuclear division, spindle microtubule attachment to the kinetochore, DNA
replication and mitotic spindle checkpoint signaling.

The interpretation is therefore not “one gene changed dramatically,” but rather:

> A coordinated proliferative/genome-maintenance transcriptional program is
> shifted toward T3.

## T1-associated processes

The T1 side contained processes involving:

- carbohydrate metabolism,
- lipid metabolic regulation,
- small-molecule catabolism,
- extracellular matrix organization,
- cell-junction biology,
- epithelial morphogenesis,
- vasculature development.

Some GO terms are broad or semantically overlapping. Terms involving synapse
organization should therefore not automatically be interpreted as evidence of
neuronal biology. Shared adhesion, membrane, cytoskeletal and signaling genes
can contribute to such annotations.

## Independent pathway view: KEGG

KEGG produced a broadly concordant picture.

Strong T3-associated pathways included:

- Ribosome
- Cell cycle
- DNA replication
- Homologous recombination
- Oxidative phosphorylation
- antigen processing and presentation
- T-cell receptor signaling

T1-associated pathways included:

- glycan degradation,
- butanoate metabolism,
- AMPK signaling,
- integrin signaling,
- focal adhesion,
- cadherin signaling,
- insulin signaling.

Disease-labelled KEGG pathways require caution. For example, enrichment of a
pathway named after a disease does not mean that the patients have that disease.
Such terms indicate that genes represented in that pathway are overrepresented
in the ranked expression signal.

## GO + KEGG convergence

GO and KEGG are not independent experiments because both are derived from the
same expression dataset.

Nevertheless, their overlap provides internal concordance.

The T3 side repeatedly converges on:

    cell division
    chromosome segregation
    DNA replication
    DNA repair
    protein synthesis
    immune-associated signaling

The T1 side repeatedly converges on:

    metabolism
    extracellular/tissue interaction
    adhesion
    integrin/focal-adhesion signaling
    metabolic signaling

This provides a more coherent interpretation than focusing on individual
nominally significant genes.

## Candidate genes

Representative candidates were selected from the pathway-level signal.

T3-associated candidates:

- KNL1
- RAD51

T1-associated candidates:

- GYS1
- INSR

Additional exploratory candidates include CDK1, CCNB1, SCD and TNC.

These are NOT genome-wide significant DEGs and are NOT being presented as
validated biomarkers.

They are representative genes whose direction and pathway context make them
useful for patient-level exploratory validation.

## Patient-level paired validation

For each candidate, the analysis compares:

    VST(T3) - VST(T1)

within each patient.

This asks whether the observed direction is reproduced across individual
paired samples rather than being driven entirely by a few patients.

Examples:

- KNL1: median paired change ≈ +0.57; 17/24 patients increased.
- GYS1: median paired change ≈ -0.28; 17/24 patients decreased.
- INSR: median paired change ≈ -0.27; 17/24 patients decreased.

Other candidates showed weaker or less consistent paired evidence.

These analyses remain exploratory because the candidates were selected after
examining the pathway results.

## What this analysis taught me

The important result of this project is not simply a list of “significant
genes.”

The analysis demonstrates a complete reasoning chain:

    paired study design
          ↓
    normalization + QC
          ↓
    variance exploration
          ↓
    paired differential expression
          ↓
    multiple-testing interpretation
          ↓
    whole-transcriptome GSEA
          ↓
    redundancy reduction
          ↓
    GO/KEGG convergence
          ↓
    representative candidate selection
          ↓
    patient-level paired validation

The strongest biological conclusion comes from the coordinated pathway signal,
while individual candidate genes remain exploratory.